# Phase 5: Sentence-BERT Training (Requirement-Aligned)

This notebook follows the required approach:
- Fine-tune a multilingual Sentence-BERT model on Sinhala harmful-content labels
- Train a classifier on Sentence-BERT embeddings for `DISINFO/HATE/NORMAL`
- Save artifacts for unseen evaluation


In [4]:
import sys
import subprocess
from pathlib import Path

req = Path("notebooks/requirements.txt")
if not req.exists():
    req = Path("../notebooks/requirements.txt")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(req)])


0

In [5]:
from pathlib import Path
import json
import math
import logging
from datetime import datetime
import pickle
import random
import re
import sys

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

from sentence_transformers import SentenceTransformer, InputExample, losses, evaluation, LoggingHandler
from importlib.metadata import version, PackageNotFoundError

REPO_ROOT = Path.cwd().resolve()
DATA_ROOT = Path('/root/separate_volume')
if not DATA_ROOT.exists():
    DATA_ROOT = REPO_ROOT

DATA_PATH = DATA_ROOT / 'datasets/splits/current/train_labeled_331.csv'
RUNS_ROOT = DATA_ROOT / 'training/artifacts/runs'
RUN_ID = datetime.now().strftime('run_%Y%m%d_%H%M%S')
RUN_ROOT = RUNS_ROOT / RUN_ID
MODEL_ROOT = RUN_ROOT / 'model'
SBERT_DIR = MODEL_ROOT / 'sbert_model'
CLF_PATH = MODEL_ROOT / 'embedding_classifier.pkl'
REPORT_DIR = RUN_ROOT / 'reports'
LATEST_RUN_FILE = RUNS_ROOT / 'latest_run.txt'

BASE_SBERT = 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2'
LABEL_ORDER = ['DISINFO', 'HATE', 'NORMAL']
VAL_SIZE = 0.15
SEED = 42
EPOCHS = 2
BATCH_SIZE = 24
MAX_SEQ_LENGTH = 192

random.seed(SEED)
np.random.seed(SEED)

logging.basicConfig(
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    level=logging.INFO,
    handlers=[LoggingHandler()],
)

def _version_tuple(v: str):
    return tuple(int(part) for part in re.findall(r'\d+', v)[:3])

try:
    accelerate_version = version('accelerate')
except PackageNotFoundError as exc:
    raise RuntimeError(
        'Missing dependency: accelerate. Run: pip install -r notebooks/requirements.txt'
    ) from exc

if _version_tuple(accelerate_version) < (0, 26, 0):
    raise RuntimeError(
        f'accelerate>={0.26} required, found {accelerate_version}. '
        'Run: pip install -r notebooks/requirements.txt'
    )

# Preflight: ensure transformers can resolve AcceleratorConfig in this live kernel.
try:
    from transformers.training_args import AcceleratorConfig  # noqa: F401
except Exception as exc:
    raise RuntimeError(
        'Kernel has stale transformers import state (AcceleratorConfig unavailable). '
        'Restart kernel (or Jupyter server) and rerun all cells from top.'
    ) from exc

print(f'[env] python={sys.executable}')
print(f'[env] accelerate={accelerate_version}')

MODEL_ROOT.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
print(f'[run] id={RUN_ID}')
print(f'[run] root={RUN_ROOT}')
DATA_PATH, SBERT_DIR


[env] python=D:\Desktop\Projects\client\sl-social-media-risk-analysis\venv\Scripts\python.exe
[env] accelerate=1.13.0
[run] id=run_20260316_232209
[run] root=D:\client-projects\sl-social-media-risk-analysis\training\artifacts\runs\run_20260316_232209


(WindowsPath('D:/client-projects/sl-social-media-risk-analysis/datasets/splits/current/train_labeled_331.csv'),
 WindowsPath('D:/client-projects/sl-social-media-risk-analysis/training/artifacts/runs/run_20260316_232209/model/sbert_model'))

In [6]:
def clean_text(text: str) -> str:
    text = "" if pd.isna(text) else str(text)
    text = text.replace("\u200d", "")
    text = " ".join(text.split())
    return text.strip()


def normalize_label(label: str) -> str:
    label = "" if pd.isna(label) else str(label)
    return re.sub(r"[,\s]+$", "", label.strip().upper())


if not DATA_PATH.exists():
    fallback = ROOT / "datasets/labeled/annotator_a_llm.csv"
    if not fallback.exists():
        raise FileNotFoundError(f"Missing both {DATA_PATH} and fallback {fallback}")
    src = pd.read_csv(fallback)
    src["annotator_label"] = src["annotator_label"].fillna("").astype(str).str.strip().str.upper()
    labeled = src[src["annotator_label"].isin(["NORMAL", "HATE", "DISINFO"])].copy()
    targets = {"NORMAL": 2850, "HATE": 2850, "DISINFO": 950}
    parts = []
    for lbl, n in targets.items():
        pool = labeled[labeled["annotator_label"] == lbl]
        if len(pool) < n:
            raise ValueError(f"Not enough rows for {lbl}: {len(pool)} < {n}")
        parts.append(pool.sample(n=n, random_state=SEED))
    built = pd.concat(parts, ignore_index=True).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    built.to_csv(DATA_PATH, index=False, encoding="utf-8")
    print(f"Created missing split: {DATA_PATH} | rows={len(built)}")

df = pd.read_csv(DATA_PATH)
text_col = "clean_text" if "clean_text" in df.columns else "text"
if text_col not in df.columns or "annotator_label" not in df.columns:
    raise ValueError("Expected columns: text/clean_text and annotator_label")

df = df.copy()
df["text"] = df[text_col].apply(clean_text)
df["label"] = df["annotator_label"].apply(normalize_label)
df = df[df["text"].str.len() > 0].copy()
df = df[df["label"].isin(LABEL_ORDER)].copy()
df = df.drop_duplicates(subset=["candidate_id"], keep="first")

label2id = {label: i for i, label in enumerate(LABEL_ORDER)}
id2label = {i: label for label, i in label2id.items()}
df["label_id"] = df["label"].map(label2id).astype(int)

train_df, val_df = train_test_split(
    df[["candidate_id", "source", "text", "label", "label_id"]],
    test_size=VAL_SIZE,
    random_state=SEED,
    stratify=df["label_id"],
)

print("Rows:", len(df))
print("Train:", len(train_df), "Val:", len(val_df))
print("Label distribution (train):")
print(train_df["label"].value_counts())


Rows: 6650
Train: 5652 Val: 998
Label distribution (train):
label
HATE       2422
NORMAL     2422
DISINFO     808
Name: count, dtype: int64


In [7]:
# 1) Fine-tune Sentence-BERT on single-sentence labeled data.
model = SentenceTransformer(BASE_SBERT)
model.max_seq_length = MAX_SEQ_LENGTH

train_examples = [
    InputExample(texts=[row.text], label=int(row.label_id))
    for row in train_df.itertuples(index=False)
]
train_loader = DataLoader(train_examples, shuffle=True, batch_size=BATCH_SIZE)
train_loss = losses.BatchSemiHardTripletLoss(
    model=model,
    margin=5,
)

steps_per_epoch = len(train_loader)
warmup_steps = math.ceil(steps_per_epoch * EPOCHS * 0.1)
eval_steps = max(50, steps_per_epoch // 4)
start_time = datetime.now()
print(f"[train] start={start_time.isoformat(timespec='seconds')} epochs={EPOCHS} steps_per_epoch={steps_per_epoch} warmup_steps={warmup_steps} loss=BatchSemiHardTripletLoss")

import importlib
import transformers.utils.import_utils as tf_import_utils
importlib.reload(tf_import_utils)
from transformers.utils import is_accelerate_available
acc_ok = is_accelerate_available()
print(f"[env] transformers_accelerate_available={acc_ok}")
if not acc_ok:
    raise RuntimeError(
        "Transformers cannot import accelerate in this running kernel. "
        "Run `python -m pip install -r notebooks/requirements.txt` with this kernel Python, then restart kernel and run all cells."
    )

model.fit(
    train_objectives=[(train_loader, train_loss)],
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    output_path=str(SBERT_DIR),
    show_progress_bar=True,
)

end_time = datetime.now()
print(f"[train] end={end_time.isoformat(timespec='seconds')} duration={end_time - start_time}")
print("Saved fine-tuned Sentence-BERT:", SBERT_DIR)


2026-03-16 23:22:09 - INFO - Use pytorch device_name: cpu
2026-03-16 23:22:09 - INFO - Load pretrained SentenceTransformer: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
[train] start=2026-03-16T23:22:17 epochs=2 steps_per_epoch=236 warmup_steps=48 loss=BatchSemiHardTripletLoss
[env] transformers_accelerate_available=True


KeyboardInterrupt: 

In [ ]:
# 2) Train embedding classifier (required classification layer on top of SBERT embeddings).
model = SentenceTransformer(str(SBERT_DIR))

X_train = model.encode(train_df["text"].tolist(), batch_size=64, show_progress_bar=True, normalize_embeddings=True)
X_val = model.encode(val_df["text"].tolist(), batch_size=64, show_progress_bar=True, normalize_embeddings=True)
y_train = train_df["label_id"].to_numpy()
y_val = val_df["label_id"].to_numpy()

clf = LogisticRegression(
    max_iter=3000,
    class_weight="balanced",
    multi_class="multinomial",
    n_jobs=-1,
)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_val)

acc = float(accuracy_score(y_val, y_pred))
macro_f1 = float(f1_score(y_val, y_pred, average="macro"))
report = classification_report(y_val, y_pred, target_names=LABEL_ORDER, output_dict=True)
cm = confusion_matrix(y_val, y_pred).tolist()

with CLF_PATH.open("wb") as f:
    pickle.dump(clf, f)

meta = {
    "run_id": RUN_ID,
    "run_root": str(RUN_ROOT),
    "approach": "Sentence-BERT fine-tune + embedding classifier",
    "base_sbert": BASE_SBERT,
    "data_path": str(DATA_PATH),
    "label_order": LABEL_ORDER,
    "label2id": label2id,
    "id2label": {str(k): v for k, v in id2label.items()},
    "counts": {
      "all_rows": int(len(df)),
      "train_rows": int(len(train_df)),
      "val_rows": int(len(val_df))
    },
    "metrics": {
      "val_accuracy": acc,
      "val_macro_f1": macro_f1,
      "val_disinfo_recall": report.get("DISINFO", {}).get("recall", None)
    }
}
LATEST_RUN_FILE.parent.mkdir(parents=True, exist_ok=True)
LATEST_RUN_FILE.write_text(str(RUN_ROOT), encoding="utf-8")
(MODEL_ROOT / "meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
(REPORT_DIR / "phase5_sbert_val_report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
(REPORT_DIR / "phase5_sbert_val_confusion_matrix.json").write_text(
    json.dumps({"labels": LABEL_ORDER, "matrix": cm}, ensure_ascii=False, indent=2),
    encoding="utf-8"
)
val_out = val_df.copy()
val_out["y_true"] = [id2label[int(v)] for v in y_val]
val_out["y_pred"] = [id2label[int(v)] for v in y_pred]
val_out.to_csv(REPORT_DIR / "phase5_sbert_val_predictions.csv", index=False, encoding="utf-8")

print("Validation accuracy:", acc)
print("Validation macro_f1:", macro_f1)
print("DISINFO recall:", report.get("DISINFO", {}).get("recall", None))
print("Saved model artifacts:", MODEL_ROOT)
print("Latest run pointer:", LATEST_RUN_FILE)


In [ ]:
# Threshold tuning on validation set (DISINFO/HATE).
import numpy as np
from sklearn.metrics import f1_score, classification_report

probs = clf.predict_proba(X_val)
prob_cols = [f"prob_{label}" for label in LABEL_ORDER]
val_out = val_out.copy()
for idx, label in enumerate(LABEL_ORDER):
    val_out[prob_cols[idx]] = probs[:, idx]

def predict_with_thresholds(prob_row, t_dis, t_hate):
    p_dis, p_hate, p_norm = prob_row
    candidates = []
    if p_dis >= t_dis:
        candidates.append(('DISINFO', p_dis))
    if p_hate >= t_hate:
        candidates.append(('HATE', p_hate))
    if candidates:
        return max(candidates, key=lambda x: x[1])[0]
    return 'NORMAL'

best = None
grid = np.linspace(0.2, 0.8, 13)
y_true = val_out['y_true'].tolist()
for t_dis in grid:
    for t_hate in grid:
        preds = [predict_with_thresholds(p, t_dis, t_hate) for p in probs]
        f1 = f1_score(y_true, preds, average='macro')
        if best is None or f1 > best['macro_f1']:
            best = {
                't_disinfo': float(t_dis),
                't_hate': float(t_hate),
                'macro_f1': float(f1),
                'report': classification_report(y_true, preds, output_dict=True)
            }

tuned_preds = [predict_with_thresholds(p, best['t_disinfo'], best['t_hate']) for p in probs]
val_out['y_pred_tuned'] = tuned_preds
val_out.to_csv(REPORT_DIR / 'phase5_sbert_val_predictions_with_probs.csv', index=False, encoding='utf-8')
(REPORT_DIR / 'phase5_sbert_val_thresholds.json').write_text(
    json.dumps(best, ensure_ascii=False, indent=2), encoding='utf-8'
)
print('Best thresholds:', best)
